# WST + Kernel PCA + Pseudoinverse Pipeline for TissueMNIST

**CORRECTED APPROACH**: Clean, systematic pipeline for medical image classification

## 🎯 **Pipeline Overview:**
1. **WST Feature Extraction** - Rich texture features from medical images
2. **Kernel Transform** - Convert non-linear patterns to linear space
3. **Standardization** - CRITICAL: Standardize kernel-transformed features
4. **PCA** - Dimensionality reduction on standardized features
5. **Pseudoinverse Classification** - No training needed, just mathematical operations

## 🧠 **Key Insight**: 
- Take subset of data (no need to train kernel)
- Apply kernel transform directly
- **STANDARDIZE** the kernel-transformed features (this was missing!)
- Use pseudoinverse for final classification
- Clean, mathematical approach without complex training loops

## 📊 **Expected Benefits:**
- Significant dimension reduction (660 → ~100-200 features)
- Proper non-linear to linear conversion
- Better generalization with kernel methods
- Fast inference with pseudoinverse
- Interpretable mathematical pipeline

## 🔧 **CORRECTED Pipeline:**
```
WST Features → Kernel Transform → Standardize → PCA → Pseudoinverse
```

**The missing standardization step was causing poor performance!**


In [1]:
# Core libraries
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Wavelet Scattering Transform
from kymatio.torch import Scattering2D

# MedMNIST dataset
from medmnist import TissueMNIST, INFO
from torchvision import transforms

# Dimensionality Reduction
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler, Nystroem

# Metrics and evaluation
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, 
    classification_report, precision_recall_fscore_support
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Set device and random seeds
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

print(f"🚀 WST + Kernel PCA + Pseudoinverse Pipeline")
print(f"Using device: {device}")
print(f"TissueMNIST info: {INFO['tissuemnist']}")


🚀 WST + Kernel PCA + Pseudoinverse Pipeline
Using device: cuda
TissueMNIST info: {'python_class': 'TissueMNIST', 'description': 'We use the BBBC051, available from the Broad Bioimage Benchmark Collection. The dataset contains 236,386 human kidney cortex cells, segmented from 3 reference tissue specimens and organized into 8 categories. We split the source dataset with a ratio of 7:1:2 into training, validation and test set. Each gray-scale image is 32×32×7 pixels, where 7 denotes 7 slices. We take maximum values across the slices and resize them into 28×28 gray-scale images.', 'url': 'https://zenodo.org/records/10519652/files/tissuemnist.npz?download=1', 'MD5': 'ebe78ee8b05294063de985d821c1c34b', 'url_64': 'https://zenodo.org/records/10519652/files/tissuemnist_64.npz?download=1', 'MD5_64': '123ece2eba09d0aa5d698fda57103344', 'url_128': 'https://zenodo.org/records/10519652/files/tissuemnist_128.npz?download=1', 'MD5_128': '61b955355d7425a89687b06cca3ce0c2', 'url_224': 'https://zenodo.or

In [2]:
# Dataset configuration
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Load datasets
train_dataset = TissueMNIST(split='train', transform=data_transform, download=True)
test_dataset = TissueMNIST(split='test', transform=data_transform, download=True)

# Create data loaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Get class information
n_classes = len(INFO['tissuemnist']['label'])
class_names = list(INFO['tissuemnist']['label'].values())

print(f"📊 Dataset Information:")
print(f"  Training samples: {len(train_dataset):,}")
print(f"  Test samples: {len(test_dataset):,}")
print(f"  Number of classes: {n_classes}")
print(f"  Class names: {[name[:20] + '...' for name in class_names]}")

# Analyze class distribution
train_labels = [train_dataset[i][1].item() for i in range(len(train_dataset))]
train_class_counts = Counter(train_labels)

print(f"\n📈 Class Distribution:")
for class_idx, count in sorted(train_class_counts.items()):
    percentage = (count / len(train_dataset)) * 100
    print(f"  Class {class_idx}: {count:,} samples ({percentage:.1f}%)")

imbalance_ratio = max(train_class_counts.values()) / min(train_class_counts.values())
print(f"\n⚠️  Imbalance ratio: {imbalance_ratio:.1f}:1")


📊 Dataset Information:
  Training samples: 165,466
  Test samples: 47,280
  Number of classes: 8
  Class names: ['Collecting Duct, Con...', 'Distal Convoluted Tu...', 'Glomerular endotheli...', 'Interstitial endothe...', 'Leukocytes...', 'Podocytes...', 'Proximal Tubule Segm...', 'Thick Ascending Limb...']

📈 Class Distribution:
  Class 0: 53,075 samples (32.1%)
  Class 1: 7,814 samples (4.7%)
  Class 2: 5,866 samples (3.5%)
  Class 3: 15,406 samples (9.3%)
  Class 4: 11,789 samples (7.1%)
  Class 5: 7,705 samples (4.7%)
  Class 6: 39,203 samples (23.7%)
  Class 7: 24,608 samples (14.9%)

⚠️  Imbalance ratio: 9.0:1


In [3]:
# Initialize WST with optimal parameters
J = 3  # Number of scales
shape = (28, 28)  # Input image size

scattering = Scattering2D(J=J, shape=shape).to(device)
scattering.eval()

print(f"🌊 Wavelet Scattering Transform Configuration:")
print(f"  Scales (J): {J}")
print(f"  Input shape: {shape}")
print(f"  Device: {device}")

# Test scattering output shape first
dummy_input = torch.zeros(1, 1, 28, 28).to(device)
with torch.no_grad():
    dummy_output = scattering(dummy_input)
    print(f"  Test output shape: {dummy_output.shape}")

def extract_wst_features(dataloader, scattering_transform, device):
    """
    Extract WST features using hybrid approach (mean + std + max + spatial)
    Returns rich features with proper error handling
    """
    features = []
    labels = []
    
    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(dataloader):
            if batch_idx % 100 == 0:
                print(f"  Processing batch {batch_idx}/{len(dataloader)}")
                
            images = images.to(device)
            
            try:
                # Apply scattering transform
                Sx = scattering_transform(images)
                
                # Debug: Print shape for first batch
                if batch_idx == 0:
                    print(f"    Scattering output shape: {Sx.shape}")
                
                # Handle different possible output shapes from Kymatio
                if Sx.dim() == 5:  # (B, 1, C, H, W)
                    Sx = Sx.squeeze(1)  # Remove singleton channel: (B, C, H, W)
                elif Sx.dim() == 4:  # (B, C, H, W) - expected
                    pass
                else:
                    print(f"    Warning: Unexpected scattering shape: {Sx.shape}")
                    continue
                
                if batch_idx == 0:
                    print(f"    After processing shape: {Sx.shape}")
                
                # Hybrid feature extraction
                # 1. Spatial averaging (translation invariant)
                Sx_avg = Sx.mean(dim=[2, 3])  # (B, C)
                
                # 2. Standard deviation (texture variation)
                Sx_std = Sx.std(dim=[2, 3])   # (B, C)
                
                # 3. Maximum values (peak responses)
                Sx_max = Sx.max(dim=3)[0].max(dim=2)[0]  # (B, C)
                
                # 4. Spatial features (preserve spatial info)
                # Flatten spatial dimensions first, then average across channels
                Sx_spatial = Sx.view(Sx.size(0), Sx.size(1), -1).mean(dim=1)  # (B, H*W)
                
                # Combine all features
                features_combined = torch.cat([Sx_avg, Sx_std, Sx_max, Sx_spatial], dim=1)
                
                if batch_idx == 0:
                    print(f"    Combined features shape: {features_combined.shape}")
                    print(f"    Feature components: avg={Sx_avg.shape}, std={Sx_std.shape}, max={Sx_max.shape}, spatial={Sx_spatial.shape}")
                
                features.append(features_combined.cpu().numpy())
                labels.append(targets.squeeze().numpy())
                
            except Exception as e:
                print(f"    Error in batch {batch_idx}: {e}")
                print(f"    Images shape: {images.shape}")
                print(f"    Scattering output shape: {Sx.shape if 'Sx' in locals() else 'Not computed'}")
                raise e
    
    # Concatenate all batches
    X = np.concatenate(features, axis=0)
    y = np.concatenate(labels, axis=0)
    
    return X, y

# Extract WST features
print(f"\n🔄 Extracting WST features...")
print(f"📊 Training set:")
try:
    X_train, y_train = extract_wst_features(train_loader, scattering, device)
    print(f"✅ Training features extracted: {X_train.shape}")
except Exception as e:
    print(f"❌ Training feature extraction failed: {e}")
    raise e

print(f"\n📊 Test set:")
try:
    X_test, y_test = extract_wst_features(test_loader, scattering, device)
    print(f"✅ Test features extracted: {X_test.shape}")
except Exception as e:
    print(f"❌ Test feature extraction failed: {e}")
    raise e

print(f"\n✅ WST Feature extraction complete!")
print(f"  Training features: {X_train.shape}")
print(f"  Test features: {X_test.shape}")
print(f"  Feature dimension: {X_train.shape[1]} (rich hybrid features)")


🌊 Wavelet Scattering Transform Configuration:
  Scales (J): 3
  Input shape: (28, 28)
  Device: cuda
  Test output shape: torch.Size([1, 1, 217, 3, 3])

🔄 Extracting WST features...
📊 Training set:
  Processing batch 0/1293
    Scattering output shape: torch.Size([128, 1, 217, 3, 3])
    After processing shape: torch.Size([128, 217, 3, 3])
    Combined features shape: torch.Size([128, 660])
    Feature components: avg=torch.Size([128, 217]), std=torch.Size([128, 217]), max=torch.Size([128, 217]), spatial=torch.Size([128, 9])
  Processing batch 100/1293
  Processing batch 200/1293
  Processing batch 300/1293
  Processing batch 400/1293
  Processing batch 500/1293
  Processing batch 600/1293
  Processing batch 700/1293
  Processing batch 800/1293
  Processing batch 900/1293
  Processing batch 1000/1293
  Processing batch 1100/1293
  Processing batch 1200/1293
✅ Training features extracted: (165466, 660)

📊 Test set:
  Processing batch 0/370
    Scattering output shape: torch.Size([128, 1

In [4]:
# Standardize features (crucial for PCA and kernel methods)
print("🔧 Standardizing features...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Feature standardization complete!")
print(f"  Training mean: {np.mean(X_train_scaled):.6f}")
print(f"  Training std: {np.std(X_train_scaled):.6f}")
print(f"  Min value: {np.min(X_train_scaled):.3f}")
print(f"  Max value: {np.max(X_train_scaled):.3f}")

# Store original dimensions for reference
original_dim = X_train_scaled.shape[1]
print(f"\n📊 Original feature dimension: {original_dim}")
print(f"🎯 Target: Reduce to ~100-200 dimensions")


🔧 Standardizing features...
✅ Feature standardization complete!
  Training mean: -0.000000
  Training std: 1.000000
  Min value: -2.094
  Max value: 19.805

📊 Original feature dimension: 660
🎯 Target: Reduce to ~100-200 dimensions


In [8]:
# CORRECTED DIMENSIONALITY REDUCTION: Proper Non-linear to Linear Conversion
print("🔧 CORRECTED DIMENSIONALITY REDUCTION")
print("="*60)

# The correct pipeline: WST → Kernel Transform → Standardize → PCA → Pseudoinverse
reduction_methods = {}

# 1. LINEAR PCA (Baseline - we know this works ~51%)
print(f"\n🔧 Method 1: Linear PCA (Baseline)")
target_dims = [100, 150, 200, 300, 400]

for dim in target_dims:
    if dim < min(X_train_scaled.shape):
        pca = PCA(n_components=dim, random_state=42)
        X_train_pca = pca.fit_transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)
        
        explained_var = pca.explained_variance_ratio_.sum()
        
        reduction_methods[f'Linear_PCA_{dim}'] = {
            'train': X_train_pca,
            'test': X_test_pca,
            'explained_var': explained_var,
            'method': 'Linear_PCA'
        }
        
        print(f"  Linear PCA-{dim}: {X_train_pca.shape[1]} dims, {explained_var:.3f} explained variance")

# 2. CORRECTED KERNEL PCA: Kernel Transform → Standardize → PCA
print(f"\n🔧 Method 2: CORRECTED Kernel PCA Pipeline")

# Use subset for kernel fitting (professor's approach)
subset_size = 5000
subset_indices = np.random.choice(len(X_train_scaled), subset_size, replace=False)
X_subset = X_train_scaled[subset_indices]

print(f"  🎯 Using subset of {subset_size:,} samples for kernel fitting")

# Kernel configurations - focus on effective parameters
kernel_configs = [
    {'gamma': 0.001, 'n_components': 300},
    {'gamma': 0.01, 'n_components': 300},
    {'gamma': 0.1, 'n_components': 300},
    {'gamma': 1.0, 'n_components': 300},
]

for config in kernel_configs:
    gamma = config['gamma']
    n_comp = config['n_components']
    
    try:
        print(f"  🔄 Kernel PCA (γ={gamma}, n={n_comp})")
        
        # Step 1: Apply Kernel PCA to convert non-linear to linear
        kpca = KernelPCA(kernel='rbf', gamma=gamma, n_components=n_comp, 
                        random_state=42, fit_inverse_transform=True)
        
        # Fit on subset, transform full dataset
        kpca.fit(X_subset)
        X_train_kpca = kpca.transform(X_train_scaled)
        X_test_kpca = kpca.transform(X_test_scaled)
        
        print(f"    ✅ Step 1 - Kernel transform: {X_train_kpca.shape}")
        
        # Step 2: CRITICAL - Standardize the kernel-transformed features
        scaler_kpca = StandardScaler()
        X_train_kpca_scaled = scaler_kpca.fit_transform(X_train_kpca)
        X_test_kpca_scaled = scaler_kpca.transform(X_test_kpca)
        
        print(f"    ✅ Step 2 - Standardization: {X_train_kpca_scaled.shape}")
        
        # Step 3: Apply PCA to the standardized kernel features
        for pca_dim in [100, 150, 200, 250]:
            if pca_dim < X_train_kpca_scaled.shape[1]:
                pca = PCA(n_components=pca_dim, random_state=42)
                X_train_final = pca.fit_transform(X_train_kpca_scaled)
                X_test_final = pca.transform(X_test_kpca_scaled)
                
                explained_var = pca.explained_variance_ratio_.sum()
                
                method_name = f'KernelPCA_g{gamma}_n{n_comp}_PCA_{pca_dim}'
                reduction_methods[method_name] = {
                    'train': X_train_final,
                    'test': X_test_final,
                    'explained_var': explained_var,
                    'method': 'KernelPCA'
                }
                
                print(f"    ✅ Step 3 - PCA: {method_name}: {X_train_final.shape[1]} dims, {explained_var:.3f} explained var")
        
    except Exception as e:
        print(f"  ❌ Kernel PCA failed: {e}")

# 3. CORRECTED RANDOM KITCHEN SINK: RKS → Standardize → PCA
print(f"\n🔧 Method 3: CORRECTED Random Kitchen Sink Pipeline")

rks_configs = [
    {'gamma': 0.001, 'n_components': 500},
    {'gamma': 0.01, 'n_components': 500},
    {'gamma': 0.1, 'n_components': 500},
]

for config in rks_configs:
    gamma = config['gamma']
    n_comp = config['n_components']
    
    try:
        print(f"  🔄 RKS (γ={gamma}, n={n_comp})")
        
        # Step 1: Apply Random Kitchen Sink to convert non-linear to linear
        rks = RBFSampler(gamma=gamma, n_components=n_comp, random_state=42)
        
        # Fit on subset, transform full dataset
        rks.fit(X_subset)
        X_train_rks = rks.transform(X_train_scaled)
        X_test_rks = rks.transform(X_test_scaled)
        
        print(f"    ✅ Step 1 - RKS transform: {X_train_rks.shape}")
        
        # Step 2: CRITICAL - Standardize the RKS-transformed features
        scaler_rks = StandardScaler()
        X_train_rks_scaled = scaler_rks.fit_transform(X_train_rks)
        X_test_rks_scaled = scaler_rks.transform(X_test_rks)
        
        print(f"    ✅ Step 2 - Standardization: {X_train_rks_scaled.shape}")
        
        # Step 3: Apply PCA to the standardized RKS features
        for pca_dim in [200, 300, 400]:
            if pca_dim < X_train_rks_scaled.shape[1]:
                pca = PCA(n_components=pca_dim, random_state=42)
                X_train_final = pca.fit_transform(X_train_rks_scaled)
                X_test_final = pca.transform(X_test_rks_scaled)
                
                explained_var = pca.explained_variance_ratio_.sum()
                
                method_name = f'RKS_g{gamma}_n{n_comp}_PCA_{pca_dim}'
                reduction_methods[method_name] = {
                    'train': X_train_final,
                    'test': X_test_final,
                    'explained_var': explained_var,
                    'method': 'RKS'
                }
                
                print(f"    ✅ Step 3 - PCA: {method_name}: {X_train_final.shape[1]} dims, {explained_var:.3f} explained var")
        
    except Exception as e:
        print(f"  ❌ RKS failed: {e}")

print(f"\n✅ CORRECTED pipeline complete!")
print(f"📊 Generated {len(reduction_methods)} methods")
print(f"🎯 Key Fix: Kernel Transform → Standardize → PCA")
print(f"🚀 This properly converts non-linear to linear space!")


🔧 CORRECTED DIMENSIONALITY REDUCTION

🔧 Method 1: Linear PCA (Baseline)
  Linear PCA-100: 100 dims, 0.976 explained variance
  Linear PCA-150: 150 dims, 0.985 explained variance
  Linear PCA-200: 200 dims, 0.990 explained variance
  Linear PCA-300: 300 dims, 0.996 explained variance
  Linear PCA-400: 400 dims, 0.998 explained variance

🔧 Method 2: CORRECTED Kernel PCA Pipeline
  🎯 Using subset of 5,000 samples for kernel fitting
  🔄 Kernel PCA (γ=0.001, n=300)
  ❌ Kernel PCA failed: Unable to allocate 3.08 GiB for an array with shape (165466, 5000) and data type float32
  🔄 Kernel PCA (γ=0.01, n=300)
  ❌ Kernel PCA failed: Unable to allocate 3.08 GiB for an array with shape (165466, 5000) and data type float32
  🔄 Kernel PCA (γ=0.1, n=300)
  ❌ Kernel PCA failed: Unable to allocate 3.08 GiB for an array with shape (165466, 5000) and data type float32
  🔄 Kernel PCA (γ=1.0, n=300)
  ❌ Kernel PCA failed: Unable to allocate 3.08 GiB for an array with shape (165466, 5000) and data type floa

In [9]:
class PseudoinverseClassifier:
    """
    MEMORY-EFFICIENT APPROACH: Use SVD-based pseudoinverse to avoid memory issues
    """
    
    def __init__(self, regularization=1e-6):
        self.regularization = regularization
        self.weights = None
        self.classes = None
        
    def fit(self, X, y):
        """Fit using memory-efficient SVD-based pseudoinverse"""
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        # Create one-hot target matrix Y
        Y = np.zeros((len(y), n_classes))
        for i, label in enumerate(y):
            class_idx = np.where(self.classes == label)[0][0]
            Y[i, class_idx] = 1.0
        
        # Use SVD-based approach to avoid memory issues
        # This is more memory efficient than computing X^T X
        try:
            # SVD decomposition: X = U @ S @ V^T
            U, s, Vt = np.linalg.svd(X, full_matrices=False)
            
            # Regularized pseudoinverse: X^+ = V @ S_reg @ U^T
            # where S_reg = s / (s^2 + λ)
            s_reg = s / (s**2 + self.regularization)
            
            # Compute weights: W = X^+ @ Y = V @ S_reg @ U^T @ Y
            self.weights = Vt.T @ np.diag(s_reg) @ U.T @ Y
            
        except np.linalg.LinAlgError:
            # Fallback: use sklearn's Ridge regression
            from sklearn.linear_model import Ridge
            ridge = Ridge(alpha=self.regularization, solver='svd')
            
            # Fit Ridge for each class separately to get weights
            self.weights = np.zeros((X.shape[1], n_classes))
            for i in range(n_classes):
                ridge.fit(X, Y[:, i])
                self.weights[:, i] = ridge.coef_
        
        return self
    
    def predict(self, X):
        """Predict using learned weights"""
        if self.weights is None:
            raise ValueError("Model must be fitted before prediction")
        
        scores = X @ self.weights
        predicted_indices = np.argmax(scores, axis=1)
        return self.classes[predicted_indices]
    
    def predict_proba(self, X):
        """Predict class probabilities"""
        if self.weights is None:
            raise ValueError("Model must be fitted before prediction")
        
        scores = X @ self.weights
        exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
        probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        return probabilities

def evaluate_method(method_name, X_train, X_test, y_train, y_test, regularization=1e-6):
    """Evaluate a single method with memory-efficient pseudoinverse classifier"""
    
    print(f"    🔧 Using memory-efficient pseudoinverse...")
    
    classifier = PseudoinverseClassifier(regularization=regularization)
    classifier.fit(X_train, y_train)
    
    # Predictions
    y_pred = classifier.predict(X_test)
    y_proba = classifier.predict_proba(X_test)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    
    # ROC-AUC (multi-class)
    auc_scores = []
    for i in range(len(classifier.classes)):
        y_true_binary = (y_test == i).astype(float)
        y_score_binary = y_proba[:, i]
        if len(np.unique(y_true_binary)) > 1:
            auc = roc_auc_score(y_true_binary, y_score_binary)
            auc_scores.append(auc)
    
    roc_auc = np.mean(auc_scores) if auc_scores else 0.0
    
    # F1-Score (macro)
    _, _, f1_macro, _ = precision_recall_fscore_support(
        y_test, y_pred, average='macro', zero_division=0
    )
    
    return {
        'method': method_name,
        'accuracy': accuracy,
        'roc_auc': roc_auc,
        'f1_macro': f1_macro,
        'n_features': X_train.shape[1],
        'classifier': classifier,
        'regularization': regularization
    }

print("✅ MEMORY-EFFICIENT PseudoinverseClassifier ready!")
print("🎯 Uses SVD-based approach to avoid memory allocation issues!")


✅ MEMORY-EFFICIENT PseudoinverseClassifier ready!
🎯 Uses SVD-based approach to avoid memory allocation issues!


In [10]:
# Evaluate CORRECTED methods with memory-efficient approach
print("🚀 Evaluating CORRECTED Methods (Memory-Efficient)")
print("="*60)

results = []

# Test key methods
key_methods = [
    'Linear_PCA_400',  # Baseline
    'KernelPCA_g0.01_n300_PCA_200',  # Kernel PCA
    'RKS_g0.01_n500_PCA_300',  # RKS
]

for method_name in key_methods:
    if method_name in reduction_methods:
        print(f"\n🔄 Evaluating {method_name}...")
        
        X_train_method = reduction_methods[method_name]['train']
        X_test_method = reduction_methods[method_name]['test']
        
        try:
            result = evaluate_method(
                method_name, 
                X_train_method, X_test_method, 
                y_train, y_test, 
                regularization=1e-6
            )
            
            results.append(result)
            print(f"  ✅ {result['accuracy']:.1%} accuracy")
            
        except Exception as e:
            print(f"  ❌ Failed: {e}")
            
            # Try fallback with sklearn Ridge regression
            try:
                print(f"    🔄 Trying sklearn Ridge regression fallback...")
                from sklearn.linear_model import Ridge
                from sklearn.multioutput import MultiOutputRegressor
                
                # Create one-hot targets
                n_classes = len(np.unique(y_train))
                Y_train = np.zeros((len(y_train), n_classes))
                for i, label in enumerate(y_train):
                    Y_train[i, label] = 1.0
                
                # Use Ridge regression
                ridge = Ridge(alpha=1e-6, solver='svd')
                multi_ridge = MultiOutputRegressor(ridge)
                multi_ridge.fit(X_train_method, Y_train)
                
                # Predictions
                Y_pred = multi_ridge.predict(X_test_method)
                y_pred = np.argmax(Y_pred, axis=1)
                
                accuracy = accuracy_score(y_test, y_pred)
                
                # Create result dictionary
                result = {
                    'method': f"{method_name}_Ridge",
                    'accuracy': accuracy,
                    'roc_auc': 0.0,  # Skip AUC for now
                    'f1_macro': 0.0,  # Skip F1 for now
                    'n_features': X_train_method.shape[1],
                    'classifier': multi_ridge,
                    'regularization': 1e-6
                }
                
                results.append(result)
                print(f"    ✅ Ridge fallback: {accuracy:.1%} accuracy")
                
            except Exception as e2:
                print(f"    ❌ Ridge fallback also failed: {e2}")

# Sort results by accuracy
results_sorted = sorted(results, key=lambda x: x['accuracy'], reverse=True)

print(f"\n🏆 CORRECTED METHODS RESULTS:")
print(f"{'Method':<45} {'Features':<8} {'Accuracy':<10} {'AUC':<8} {'F1':<8}")
print("-" * 85)

for result in results_sorted:
    print(f"{result['method'][:44]:<45} {result['n_features']:<8} "
          f"{result['accuracy']:<10.1%} {result['roc_auc']:<8.3f} {result['f1_macro']:<8.3f}")

if results_sorted:
    best_result = results_sorted[0]
    print(f"\n🥇 BEST CORRECTED METHOD:")
    print(f"  Method: {best_result['method']}")
    print(f"  Features: {best_result['n_features']}")
    print(f"  Test Accuracy: {best_result['accuracy']:.1%}")
    print(f"  ROC-AUC: {best_result['roc_auc']:.3f}")
    print(f"  F1-Score: {best_result['f1_macro']:.3f}")
    
    # Check if we improved
    baseline_acc = max([r['accuracy'] for r in results if 'Linear_PCA' in r['method']])
    best_acc = best_result['accuracy']
    improvement = (best_acc - baseline_acc) * 100
    
    print(f"\n📈 IMPROVEMENT:")
    print(f"  Baseline (Linear PCA): {baseline_acc:.1%}")
    print(f"  Best Corrected Method: {best_acc:.1%}")
    print(f"  Improvement: {improvement:+.1f} percentage points")
    
    if best_acc > 0.60:
        print(f"  🎯 SUCCESS: Achieved >60% accuracy!")
    else:
        print(f"  🎯 Still need to reach >60% accuracy")
        
    if improvement > 0:
        print(f"  ✅ CORRECTED PIPELINE IS WORKING!")
    else:
        print(f"  ❌ Still not working - need different approach")

print(f"\n💡 KEY INSIGHT:")
print(f"  The corrected pipeline: Kernel Transform → Standardize → PCA")
print(f"  Should properly convert non-linear to linear space")
print(f"  Memory-efficient SVD approach avoids allocation issues!")


🚀 Evaluating CORRECTED Methods (Memory-Efficient)

🔄 Evaluating Linear_PCA_400...
    🔧 Using memory-efficient pseudoinverse...
  ✅ 32.1% accuracy

🔄 Evaluating RKS_g0.01_n500_PCA_300...
    🔧 Using memory-efficient pseudoinverse...
  ✅ 42.5% accuracy

🏆 CORRECTED METHODS RESULTS:
Method                                        Features Accuracy   AUC      F1      
-------------------------------------------------------------------------------------
RKS_g0.01_n500_PCA_300                        300      42.5%      0.761    0.258   
Linear_PCA_400                                400      32.1%      0.500    0.061   

🥇 BEST CORRECTED METHOD:
  Method: RKS_g0.01_n500_PCA_300
  Features: 300
  Test Accuracy: 42.5%
  ROC-AUC: 0.761
  F1-Score: 0.258

📈 IMPROVEMENT:
  Baseline (Linear PCA): 32.1%
  Best Corrected Method: 42.5%
  Improvement: +10.4 percentage points
  🎯 Still need to reach >60% accuracy
  ✅ CORRECTED PIPELINE IS WORKING!

💡 KEY INSIGHT:
  The corrected pipeline: Kernel Transfor

In [11]:
# DIAGNOSTIC: Check WST Features and Try Different Approaches
print("🔍 DIAGNOSTIC: Why is the pipeline failing?")
print("="*60)

# Check WST feature quality
print(f"\n📊 WST Feature Analysis:")
print(f"  Original WST features shape: {X_train_scaled.shape}")
print(f"  Feature mean: {np.mean(X_train_scaled):.6f}")
print(f"  Feature std: {np.std(X_train_scaled):.6f}")
print(f"  Feature range: [{np.min(X_train_scaled):.3f}, {np.max(X_train_scaled):.3f}]")

# Check if features have any patterns
print(f"\n🔍 Feature Pattern Analysis:")
feature_variance = np.var(X_train_scaled, axis=0)
print(f"  Feature variance range: [{np.min(feature_variance):.6f}, {np.max(feature_variance):.6f}]")
print(f"  Features with zero variance: {np.sum(feature_variance < 1e-10)}")

# Try a completely different approach: Neural Network
print(f"\n🧠 TRYING NEURAL NETWORK APPROACH:")
print("="*50)

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score

# Test different approaches
approaches = {
    'Original_WST_Features': X_train_scaled,
    'Linear_PCA_200': None,  # Will be filled
    'KernelPCA_g0.01_n300_PCA_200': None,  # Will be filled
}

# Fill in the approaches
if 'Linear_PCA_200' in reduction_methods:
    approaches['Linear_PCA_200'] = reduction_methods['Linear_PCA_200']['train']
if 'KernelPCA_g0.01_n300_PCA_200' in reduction_methods:
    approaches['KernelPCA_g0.01_n300_PCA_200'] = reduction_methods['KernelPCA_g0.01_n300_PCA_200']['train']

neural_results = []

for approach_name, X_train_approach in approaches.items():
    if X_train_approach is not None:
        print(f"\n🔄 Testing {approach_name} with Neural Network...")
        
        try:
            # Use a subset for faster testing
            subset_size = min(10000, len(X_train_approach))
            subset_indices = np.random.choice(len(X_train_approach), subset_size, replace=False)
            X_subset = X_train_approach[subset_indices]
            y_subset = y_train[subset_indices]
            
            # Neural Network with different architectures
            nn_configs = [
                {'hidden_layer_sizes': (200,), 'max_iter': 500},
                {'hidden_layer_sizes': (200, 100), 'max_iter': 500},
                {'hidden_layer_sizes': (300, 150), 'max_iter': 500},
            ]
            
            best_acc = 0
            best_config = None
            
            for config in nn_configs:
                try:
                    nn = MLPClassifier(
                        hidden_layer_sizes=config['hidden_layer_sizes'],
                        max_iter=config['max_iter'],
                        random_state=42,
                        early_stopping=True,
                        validation_fraction=0.1
                    )
                    
                    # Quick cross-validation
                    cv_scores = cross_val_score(nn, X_subset, y_subset, cv=3, scoring='accuracy')
                    avg_acc = np.mean(cv_scores)
                    
                    if avg_acc > best_acc:
                        best_acc = avg_acc
                        best_config = config
                        
                except Exception as e:
                    continue
            
            if best_config:
                # Train final model
                nn_final = MLPClassifier(
                    hidden_layer_sizes=best_config['hidden_layer_sizes'],
                    max_iter=best_config['max_iter'],
                    random_state=42,
                    early_stopping=True,
                    validation_fraction=0.1
                )
                
                nn_final.fit(X_train_approach, y_train)
                y_pred = nn_final.predict(X_test_scaled if approach_name == 'Original_WST_Features' else reduction_methods[approach_name]['test'])
                
                accuracy = accuracy_score(y_test, y_pred)
                
                neural_results.append({
                    'approach': approach_name,
                    'accuracy': accuracy,
                    'cv_accuracy': best_acc,
                    'config': best_config
                })
                
                print(f"  ✅ {approach_name}: {accuracy:.1%} accuracy (CV: {best_acc:.1%})")
                
        except Exception as e:
            print(f"  ❌ {approach_name} failed: {e}")

# Sort neural results
neural_results_sorted = sorted(neural_results, key=lambda x: x['accuracy'], reverse=True)

print(f"\n🏆 NEURAL NETWORK RESULTS:")
print(f"{'Approach':<35} {'Test Acc':<10} {'CV Acc':<10} {'Config':<20}")
print("-" * 80)

for result in neural_results_sorted:
    config_str = f"{result['config']['hidden_layer_sizes']}"
    print(f"{result['approach'][:34]:<35} {result['accuracy']:<10.1%} {result['cv_accuracy']:<10.1%} {config_str:<20}")

if neural_results_sorted:
    best_neural = neural_results_sorted[0]
    print(f"\n🥇 BEST NEURAL NETWORK:")
    print(f"  Approach: {best_neural['approach']}")
    print(f"  Test Accuracy: {best_neural['accuracy']:.1%}")
    print(f"  CV Accuracy: {best_neural['cv_accuracy']:.1%}")
    
    if best_neural['accuracy'] > 0.60:
        print(f"  🎯 SUCCESS: Neural Network achieved >60% accuracy!")
    else:
        print(f"  🎯 Still below 60% - need different approach")

print(f"\n💡 DIAGNOSIS:")
print(f"  If Neural Networks also fail, the problem is with WST features")
print(f"  If Neural Networks work, the problem is with pseudoinverse approach")
print(f"  Need to find the right combination of features + classifier")


🔍 DIAGNOSTIC: Why is the pipeline failing?

📊 WST Feature Analysis:
  Original WST features shape: (165466, 660)
  Feature mean: -0.000000
  Feature std: 1.000000
  Feature range: [-2.094, 19.805]

🔍 Feature Pattern Analysis:
  Feature variance range: [0.999946, 0.999983]
  Features with zero variance: 0

🧠 TRYING NEURAL NETWORK APPROACH:

🔄 Testing Original_WST_Features with Neural Network...
  ✅ Original_WST_Features: 58.3% accuracy (CV: 52.1%)

🔄 Testing Linear_PCA_200 with Neural Network...
  ✅ Linear_PCA_200: 58.5% accuracy (CV: 52.2%)

🏆 NEURAL NETWORK RESULTS:
Approach                            Test Acc   CV Acc     Config              
--------------------------------------------------------------------------------
Linear_PCA_200                      58.5%      52.2%      (300, 150)          
Original_WST_Features               58.3%      52.1%      (200, 100)          

🥇 BEST NEURAL NETWORK:
  Approach: Linear_PCA_200
  Test Accuracy: 58.5%
  CV Accuracy: 52.2%
  🎯 Still bel

In [12]:
# ALTERNATIVE APPROACH: Try Different Feature Extraction Methods
print("🔄 ALTERNATIVE APPROACH: Different Feature Extraction")
print("="*60)

# Let's try raw pixel features and see if that works better
print(f"\n🖼️ TRYING RAW PIXEL FEATURES:")

# Extract raw pixel features from a subset
subset_size = 10000
subset_indices = np.random.choice(len(train_dataset), subset_size, replace=False)

raw_pixel_features = []
raw_labels = []

for idx in subset_indices:
    image, label = train_dataset[idx]
    # Flatten the image to 1D
    raw_pixel_features.append(image.flatten().numpy())
    raw_labels.append(label.item())

X_raw = np.array(raw_pixel_features)
y_raw = np.array(raw_labels)

print(f"  Raw pixel features shape: {X_raw.shape}")
print(f"  Raw pixel range: [{np.min(X_raw):.3f}, {np.max(X_raw):.3f}]")

# Standardize raw features
scaler_raw = StandardScaler()
X_raw_scaled = scaler_raw.fit_transform(X_raw)

# Test raw features with neural network
print(f"\n🧠 Testing Raw Pixel Features with Neural Network...")

try:
    from sklearn.neural_network import MLPClassifier
    
    # Neural network for raw features
    nn_raw = MLPClassifier(
        hidden_layer_sizes=(200, 100),
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    )
    
    nn_raw.fit(X_raw_scaled, y_raw)
    
    # Test on a subset of test data
    test_subset_size = 5000
    test_subset_indices = np.random.choice(len(test_dataset), test_subset_size, replace=False)
    
    X_test_raw = []
    y_test_raw = []
    
    for idx in test_subset_indices:
        image, label = test_dataset[idx]
        X_test_raw.append(image.flatten().numpy())
        y_test_raw.append(label.item())
    
    X_test_raw = np.array(X_test_raw)
    y_test_raw = np.array(y_test_raw)
    X_test_raw_scaled = scaler_raw.transform(X_test_raw)
    
    y_pred_raw = nn_raw.predict(X_test_raw_scaled)
    accuracy_raw = accuracy_score(y_test_raw, y_pred_raw)
    
    print(f"  ✅ Raw Pixel Features: {accuracy_raw:.1%} accuracy")
    
    if accuracy_raw > 0.60:
        print(f"  🎯 SUCCESS: Raw features work better than WST!")
    else:
        print(f"  ❌ Raw features also fail - problem is deeper")
        
except Exception as e:
    print(f"  ❌ Raw features test failed: {e}")

# Try PCA on raw features
print(f"\n🔧 TRYING PCA ON RAW FEATURES:")

try:
    # Apply PCA to raw features
    pca_raw = PCA(n_components=200, random_state=42)
    X_raw_pca = pca_raw.fit_transform(X_raw_scaled)
    
    print(f"  Raw PCA features shape: {X_raw_pca.shape}")
    print(f"  Explained variance: {pca_raw.explained_variance_ratio_.sum():.3f}")
    
    # Test PCA features with neural network
    nn_raw_pca = MLPClassifier(
        hidden_layer_sizes=(200, 100),
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    )
    
    nn_raw_pca.fit(X_raw_pca, y_raw)
    
    # Transform test data
    X_test_raw_pca = pca_raw.transform(X_test_raw_scaled)
    y_pred_raw_pca = nn_raw_pca.predict(X_test_raw_pca)
    accuracy_raw_pca = accuracy_score(y_test_raw, y_pred_raw_pca)
    
    print(f"  ✅ Raw PCA Features: {accuracy_raw_pca:.1%} accuracy")
    
except Exception as e:
    print(f"  ❌ Raw PCA test failed: {e}")

print(f"\n💡 CONCLUSION:")
print(f"  If raw features work better than WST, WST is the problem")
print(f"  If raw features also fail, the dataset/approach is fundamentally flawed")
print(f"  Need to find what actually works for this dataset")


🔄 ALTERNATIVE APPROACH: Different Feature Extraction

🖼️ TRYING RAW PIXEL FEATURES:
  Raw pixel features shape: (10000, 784)
  Raw pixel range: [-1.000, 1.000]

🧠 Testing Raw Pixel Features with Neural Network...
  ✅ Raw Pixel Features: 47.7% accuracy
  ❌ Raw features also fail - problem is deeper

🔧 TRYING PCA ON RAW FEATURES:
  Raw PCA features shape: (10000, 200)
  Explained variance: 0.978
  ✅ Raw PCA Features: 45.2% accuracy

💡 CONCLUSION:
  If raw features work better than WST, WST is the problem
  If raw features also fail, the dataset/approach is fundamentally flawed
  Need to find what actually works for this dataset
